# Design for Multi-Model Approach

This document outlines the general approach and ideas behind a multi-model approach to tackling The Legend of Zelda.

The decision on whether to use equipment and the sword in particular directions is one that can be trained with neural networks and a standard training approach (i.e. without Reinforcement Learning).  The challenge is collecting enough training data.

By setting Link up in various situations, we can use save states to test whether attacking in every direction with every item results in enemy damage or kills.  The trick will be to do this efficiently with enough varied scenarios.  By running any agent through the game (including our previously trained RL trained agent), we can pause the simulation at any time and capture this data.  We can also reduce the amount of data we need to collect by smartly determining enemy locations.  E.G. if no enemies are to the West of Link, and using the sword does not resort in hitting anything (to account for overlap), then we do not have to test beams or arrows to the west.

The idea is that each model or output head of a larger model will tell us what the expected damage is for using a weapon in each of the cardinal directions.  We can then train an independent model with reinforcement learning for pathfinding and another "Agent" model for synthesizing all of the model outputs and selecting an action.

This allows us to independently refine and improve our item models and seamlessly plug them into the Reinforcement Learning based models.

## Model Kinds

The possible input to models will consist of vectors of data about item locations, enemy locations, projectile locations, image data of the screen, and goal related data such as what next rooms the model should be moving towards.

The *actual* input to a model isn't fully specified here, since we'll be figuring out which is actually the best model architecture (including input) through trial and error.

### Item Models

ItemModels are multi-headed with shared base layers.  They output an expected damage value for each cardinal direction.  That is, if we used an Arrow in the North direction, how much damage do we expect to do?  This allows each ItemModel head to tell the Agent model how much it thinks each use of it might be worth.

We will have two separate ItemModel instances that do not share any layers.  One for projectiles (arrows, boomerang, sword beams) and one for close-range items (sword, bombs).

### Movement Model

The MovementModel chooses what is the best direction to move in (N, S, E, W), and it cannot select not to move.  It will receive an additional input vector of True/False for each N, S, E, W rooms surrounding this one indicating what rooms it should be moving towards.  This can have no direction set True if the goal is to kill enemies or collect items in the current room.

It would be possible to build this as either an independent model or as part of the Agent model.  It will be a lot harder to train via reinforcement learning as a standalone model since movement is interleaved with item actions.  Moving into proximity of an enemy for an attack is a positive, but the actual attack reward wouldn't flow backwards through the gamma rate unless we somehow added this to the training.

Still, having it as a separate model means we could refine the model independently of everything else.

### Agent Model

The central, controlling model is the Agent.  It takes in as input the result of all other models as well as the input vectors/image.  All other models are intended to be highly specialized, answering a small question about the world.  The Agent is supposed to make more long-term focused decisions.  This will be trained through reinforcement learning.

## Data captured for training

In order to train most of these models, we need to capture a lot of data about the game that can be used as test or training data.  We already have an agent capable of playing the game (and can play the game ourselves).  We want to periodically pause the game and perform every possible action that can be performed and record the results.  We accomplish this through save states.

### Image

We will capture a full image of the screen.

### Action Results

We will perform each possible action in each cardinal direction (Sword North, Sword South, Bomb North, Bomb East, Move West, etc).  We then record the result of that action, such as the amount of damage done, whether we take damage, items collected, our new coordinate position, etc.

### Object States (Link, Enemies, Items, Projectiles)

We will additionally save off information about Link, Enemies, Items, and Projectiles.

**Link**

- Position, direction facing, whether we have the Clock power.
- Heart total, current hearts, selected item, sword type, shield type. (Not all equipment or triforce, we are only looking for what can be seen onscreen).

**Items**

- Item ID
- Position

**Enemies**

- Enemy ID, status (stunned/etc)
- Previous Position, Position, Next Position (in N frames), direction facing.
- Health.

**Projectiles**

- Projectile ID
- Previous Position, Position, Next Position